# Portfolio Diversification

A common recipe in quantum finance is to turn a *market graph* — assets as nodes, pairwise
return correlation as edge weights — into a combinatorial optimization problem: **Max-Cut**.
Maximizing the cut pushes strongly (positively) correlated pairs of assets onto *opposite*
sides of the partition, since separating them is what earns the objective the most weight;
weakly- or negatively-correlated pairs are cheap to leave on the *same* side. The upshot is
that each side of the resulting partition is, on average, a more mutually-diversified basket
of assets than a randomly chosen one of the same size.

This notebook builds the whole pipeline from scratch, with **no external files or datasets**
— a small synthetic multi-sector market is simulated directly below — and solves the
resulting Max-Cut problem two ways:

1. **QAOA** on a small, 8-asset toy market graph. One qubit per asset, so this only scales to
   a handful of names — but it is exact enough to check against a brute-force reference.
2. **PCE** (Pauli Correlation Encoding) on the full 50-asset market. PCE encodes many binary
   decision variables per qubit via higher-order Pauli correlators, so the same 50-node
   Max-Cut problem that would need 50 qubits for QAOA fits in a handful of qubits here.


In [ ]:
import itertools
import time
from collections import Counter
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qarp import config
from qarp.algorithms import (
    PCE,
    QAOA,
    Sampler,
    calculate_qubits,
    classical_function_max_cut,
)
from qarp.blocks import HEABlock
from qarp.engines import QarpEngine
from qarp.graphs import Graph
from qarp.optimizers import ScipyOptimizer
from qarp.plotting import plot_histogram

RNG_SEED = 42
config.seed = RNG_SEED
rng = np.random.default_rng(RNG_SEED)

## 1. A synthetic market — built here, no CSV needed

Fifty tickers across five sectors. Each asset's return is driven mostly by its **sector**
factor (so same-sector assets move together, correlation ~0.8) plus a much smaller shared
market factor (`beta`, sign-flipped for `GOLD`) and idiosyncratic noise — a standard
multi-factor return model, tuned so that correlation is concentrated *within* a sector and
close to zero *across* sectors. Prices are just the exponentiated cumulative returns,
starting at 100.

In [ ]:
N_DAYS = 500
SECTORS = {
    "TECH": dict(n_assets=12, beta=0.15, sector_vol=0.014, idio_vol=0.006),
    "ENRG": dict(n_assets=10, beta=0.15, sector_vol=0.013, idio_vol=0.006),
    "FINL": dict(n_assets=10, beta=0.15, sector_vol=0.013, idio_vol=0.006),
    "HLTH": dict(n_assets=10, beta=0.15, sector_vol=0.012, idio_vol=0.006),
    "GOLD": dict(n_assets=8, beta=-0.10, sector_vol=0.012, idio_vol=0.006),
}

market_factor = rng.normal(0.0003, 0.006, N_DAYS)
tickers, return_cols, sector_of = [], [], {}
for sector, cfg in SECTORS.items():
    sector_factor = rng.normal(0.0, cfg["sector_vol"], N_DAYS)
    for i in range(cfg["n_assets"]):
        ticker = f"{sector}{i + 1:02d}"
        idio = rng.normal(0.0, cfg["idio_vol"], N_DAYS)
        r = cfg["beta"] * market_factor + sector_factor + idio
        tickers.append(ticker)
        return_cols.append(r)
        sector_of[ticker] = sector

returns = pd.DataFrame(np.array(return_cols).T, columns=tickers)
prices = 100.0 * np.exp(returns.cumsum())
N_ASSETS = len(tickers)
print(f"Synthetic universe: {N_ASSETS} assets across {len(SECTORS)} sectors, {N_DAYS} trading days")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for t in ["TECH01", "ENRG01", "FINL01", "HLTH01", "GOLD01"]:
    ax.plot(prices.index, prices[t], label=t, lw=1.2)
ax.set_title("Simulated price paths (one representative asset per sector)")
ax.set_xlabel("trading day"); ax.set_ylabel("price")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. From correlations to a market graph

`qarp.graphs.Graph` extends `networkx.Graph`; QAOA and PCE both read node labels directly as
qubit indices, so nodes must be the contiguous integers `0 .. n_assets - 1` — asset `i` is
node `i`. Edge weights are the pairwise return correlations; pairs below `threshold` in
magnitude are dropped both to keep the Hamiltonian smaller and to prune sampling noise —
at `threshold=0.15` only genuinely same-sector pairs survive here, so the graph is (close
to) five disjoint near-cliques, one per sector.

In [ ]:
def build_market_graph(returns_df, threshold=0.15):
    """Nodes = assets (qubit indices 0..n-1), edges weighted by return correlation."""
    corr = returns_df.corr()
    cols = list(returns_df.columns)
    graph = Graph()
    graph.add_nodes_from(range(len(cols)))
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            w = corr.iloc[i, j]
            if abs(w) >= threshold:
                graph.add_edge(i, j, weight=float(w))
    return graph, cols, corr


market_graph, mg_tickers, corr = build_market_graph(returns, threshold=0.15)
print(
    f"Market graph: {market_graph.number_of_nodes()} nodes, "
    f"{market_graph.number_of_edges()} weighted edges"
)

## 3. Diversification metrics

Two small helpers used throughout: `mean_corr` averages pairwise correlation over a subset
of assets, and `diversified_and_complement` reads off the two sides of a Max-Cut bitstring
and labels the *less* mutually-correlated side as the "diversified" basket — Max-Cut only
guarantees the **total** correlation left uncut is small, not that it splits evenly between
the two sides, so this is a cheap, honest post-processing step rather than an assumption.

In [ ]:
def mean_corr(corr, idx):
    idx = list(idx)
    if len(idx) < 2:
        return float("nan")
    sub = corr.to_numpy()[np.ix_(idx, idx)]
    iu = np.triu_indices(len(idx), k=1)
    return float(sub[iu].mean())


def groups_from_bitstring(bitstring):
    idx0 = [i for i, b in enumerate(bitstring) if b == 0]
    idx1 = [i for i, b in enumerate(bitstring) if b == 1]
    return idx0, idx1


def diversified_and_complement(corr, bitstring):
    idx0, idx1 = groups_from_bitstring(bitstring)
    c0, c1 = mean_corr(corr, idx0), mean_corr(corr, idx1)
    return (idx0, idx1, c0, c1) if c0 <= c1 else (idx1, idx0, c1, c0)


def random_same_size_baseline(corr, k, n_assets, trials=2000, rng=None):
    rng = rng or np.random.default_rng(0)
    vals = [mean_corr(corr, rng.choice(n_assets, size=k, replace=False)) for _ in range(trials)]
    return float(np.mean(vals))


def annualized_vol(returns_df, idx, trading_days=252):
    idx = list(idx)
    w = np.ones(len(idx)) / len(idx)
    cov = returns_df.iloc[:, idx].cov().to_numpy()
    return float(np.sqrt((w @ cov @ w) * trading_days))


whole_corr = mean_corr(corr, range(N_ASSETS))
print(f"Whole-universe average pairwise correlation: {whole_corr:.3f}")

## 4. Toy example — QAOA on 8 assets

One qubit per asset, so this stays small: two tickers from each of `TECH`, `ENRG`, `FINL`
and `GOLD`. Small enough (`2^8 = 256` candidate bitstrings) to brute-force the true Max-Cut
optimum as an independent reference, so we can check QAOA actually found it rather than just
*a* solution.

In [ ]:
toy_tickers = ["TECH01", "TECH02", "ENRG01", "ENRG02", "FINL01", "FINL02", "GOLD01", "GOLD02"]
toy_graph, toy_tickers_ordered, toy_corr = build_market_graph(returns[toy_tickers], threshold=0.0)
print(f"Toy market graph: {toy_graph.number_of_nodes()} assets, {toy_graph.number_of_edges()} edges")
print("Node -> ticker:", dict(enumerate(toy_tickers_ordered)))
toy_graph.plot(figsize=(4, 3))

In [ ]:
def brute_force_max_cut(graph, n_nodes):
    best_val, best_bits = -np.inf, None
    for bits in itertools.product([0, 1], repeat=n_nodes):
        val = classical_function_max_cut(graph, list(bits))
        if val > best_val:
            best_val, best_bits = val, list(bits)
    return best_val, best_bits


opt_val, opt_bits = brute_force_max_cut(toy_graph, len(toy_tickers_ordered))
print(f"Brute-force optimum (independent reference): cut = {opt_val:.3f}, bits = {opt_bits}")

In [ ]:
n_layers = 3
qaoa = QAOA(
    problem=toy_graph,
    n_layers=n_layers,
    use_rzz=True,
    verbose=True,
    initial_parameters=[0.3] * 2 * n_layers,
).build()
fun, x = qaoa.run()

Sample the optimized circuit to read off the most likely partition, exactly as in
`mwe_qaoa_max_cut.ipynb`.

In [ ]:
# The ket bound at the optimum; run() already maps result.x onto the sorted
# symbol tuple (§17), so no hand-zip of symbols against x is needed.
circ = deepcopy(qaoa.get_final_state_block())
circ.measure([(q, q) for q in range(circ.n_qubits)])

engine = QarpEngine(seed=RNG_SEED)  # explicit seed: qarpx's shot-sampling RNG is
                                     # independent of config.seed
sampler = Sampler(ket=circ, n_shots=10000)
engine.build([sampler])
engine.run({})
probs = sampler.result

plot_histogram(probs, top_k=8, sort_by_prob=True,
                title="QAOA sampled solutions (toy 8-asset market graph)")

In [ ]:
qaoa_bits = list(max(probs, key=probs.get))
qaoa_val = classical_function_max_cut(toy_graph, qaoa_bits)
print(f"QAOA best sampled cut: {qaoa_val:.3f}  "
      f"(brute-force optimum: {opt_val:.3f}, ratio={qaoa_val / opt_val:.2f})")

div_idx, comp_idx, div_c, comp_c = diversified_and_complement(toy_corr, qaoa_bits)
baseline = random_same_size_baseline(toy_corr, len(div_idx), len(toy_tickers_ordered))
print(f"\nDiversified toy basket ({len(div_idx)} assets): mean pairwise corr = {div_c:.3f}")
print(f"  vs whole toy universe: {mean_corr(toy_corr, range(8)):.3f}")
print(f"  vs a size-matched random draw: {baseline:.3f}")
print("Diversified basket:", [toy_tickers_ordered[i] for i in div_idx])
print("Complementary basket:", [toy_tickers_ordered[i] for i in comp_idx])

toy_colors = {i: ("#38bdf8" if i in div_idx else "#f97316") for i in range(len(toy_tickers_ordered))}
toy_graph.plot(highlight_nodes=toy_colors, figsize=(4, 3))

The diversified basket's assets (blue) are cheaper to keep on the same side of the cut —
their pairwise correlations were low or negative to begin with — while the strongly
correlated pairs were pushed apart across the cut.

## 5. Large-scale example — PCE on the full 50-asset market

QAOA's one-qubit-per-asset encoding does not scale here: 50 binary decision variables would
need 50 qubits. PCE instead encodes them into order-`k` Pauli correlators (`Z`, `X`, `Y`
strings of weight `k`) measured on a handful of qubits, via `calculate_qubits`.

In [ ]:
order = 3
n_qubits = calculate_qubits(N_ASSETS, order, merging=True)
print(f"{N_ASSETS} assets encoded in {n_qubits} qubits "
      f"(order={order} Pauli correlators, merging=True) -- QAOA's encoding would need {N_ASSETS} qubits.")

Two classical reference points, since exhaustive search is no longer possible at this size:
a greedy local-search heuristic (repeated single-asset flips from several random starts) and
the average cut of a purely random balanced split. PCE's own optimization landscape is
non-convex, so it is also run from several random initial parameter draws, keeping the best
— a handful of restarts costs seconds here since each PCE circuit only touches
`n_qubits` qubits.

In [ ]:
def greedy_local_search_max_cut(graph, n_nodes, n_restarts=20, rng=None):
    rng = rng or np.random.default_rng(1)
    best_val, best_bits = -np.inf, None
    for _ in range(n_restarts):
        bits = list(rng.integers(0, 2, n_nodes))
        val = classical_function_max_cut(graph, bits)
        improved = True
        while improved:
            improved = False
            for i in range(n_nodes):
                bits[i] ^= 1
                new_val = classical_function_max_cut(graph, bits)
                if new_val > val:
                    val = new_val
                    improved = True
                else:
                    bits[i] ^= 1
        if val > best_val:
            best_val, best_bits = val, bits[:]
    return best_val, best_bits


def random_baseline_cut(graph, n_nodes, trials=300, rng=None):
    rng = rng or np.random.default_rng(0)
    return float(np.mean([classical_function_max_cut(graph, list(rng.integers(0, 2, n_nodes)))
                           for _ in range(trials)]))


PCE_ENGINE_SEED = 1000  # decoupled from RNG_SEED: separate stream for the shot-sampling RNG


def run_pce_multistart(graph, order, n_qubits, n_restarts=10, seed=RNG_SEED):
    """Several random initial-parameter draws; PCE's landscape is non-convex, so keep the best."""
    ansatz_rng = np.random.default_rng(seed)
    best_val, best_sol, best_pce = -np.inf, None, None
    for i in range(n_restarts):
        he_wfn = HEABlock(n_qubits, 3, True, True, True, False).build()
        init = list(ansatz_rng.uniform(0, 2 * np.pi, len(he_wfn.symbols)))
        pce = PCE(
            graph=graph, order=order, ket=he_wfn, primitive=Sampler(), merging=True,
            initial_parameters=init, verbose=False, optimizer=ScipyOptimizer("COBYQA"),
            # explicit seed: qarpx's shot-sampling RNG is independent of config.seed
            engine=QarpEngine(seed=PCE_ENGINE_SEED + i),
        ).build()
        _, _, sol = pce.run()
        val = classical_function_max_cut(graph, sol)
        if val > best_val:
            best_val, best_sol, best_pce = val, sol, pce
    return best_val, best_sol, best_pce


t0 = time.perf_counter()
greedy_val, greedy_bits = greedy_local_search_max_cut(market_graph, N_ASSETS)
rand_cut = random_baseline_cut(market_graph, N_ASSETS)
pce_val, pce_sol, pce = run_pce_multistart(market_graph, order, n_qubits, n_restarts=10)
print(f"(benchmarks + PCE multistart computed in {time.perf_counter() - t0:.1f}s)")

print("\nMax-cut objective (total weighted correlation separated across the cut):")
print(f"  random split                    : {rand_cut:6.1f}")
print(f"  PCE, best of 10 restarts, {n_qubits} qubits : {pce_val:6.1f}  "
      f"({100 * pce_val / greedy_val:.0f}% of the classical heuristic)")
print(f"  greedy local search (classical) : {greedy_val:6.1f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
fig.suptitle("PCE convergence (best restart)")
iters = list(range(len(pce.history)))
ax1.plot(iters, [i[0] for i in pce.history])
ax1.set_ylabel("loss"); ax1.set_xlabel("iteration"); ax1.grid(alpha=0.3)
ax2.plot(iters, [i[1] for i in pce.history])
ax2.set_ylabel("graph-cut value"); ax2.set_xlabel("iteration"); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Reading off the diversified portfolio

Same post-processing as the toy example: take the less mutually-correlated side of the
partition as the diversified basket, and compare it against a random draw of the same size
— both in average pairwise correlation and in realized (equal-weight) portfolio
volatility.

In [ ]:
div_idx, comp_idx, div_c, comp_c = diversified_and_complement(corr, pce_sol)
baseline = random_same_size_baseline(corr, len(div_idx), N_ASSETS)

print(f"Diversified cluster  : {len(div_idx):>2d} assets, mean pairwise corr = {div_c:.3f}")
print(f"Complementary cluster: {len(comp_idx):>2d} assets, mean pairwise corr = {comp_c:.3f}")
print(f"Whole-universe average: {whole_corr:.3f}  |  size-matched random draw: {baseline:.3f}")

div_vol = annualized_vol(returns, div_idx)
random_vols = [
    annualized_vol(returns, np.random.default_rng(s).choice(N_ASSETS, len(div_idx), replace=False))
    for s in range(200)
]
print(f"\nAnnualized volatility, equal-weight diversified basket: {div_vol * 100:.2f}%")
print(f"Annualized volatility, size-matched random baskets (avg of 200 draws): "
      f"{np.mean(random_vols) * 100:.2f}%")

print("\nSector composition:")
print("  diversified :", Counter(sector_of[mg_tickers[i]] for i in div_idx))
print("  complement  :", Counter(sector_of[mg_tickers[i]] for i in comp_idx))

In [ ]:
order_idx = div_idx + comp_idx
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(corr.to_numpy()[np.ix_(order_idx, order_idx)], cmap="RdBu_r", vmin=-1, vmax=1)
ax.axvline(len(div_idx) - 0.5, color="k", lw=1)
ax.axhline(len(div_idx) - 0.5, color="k", lw=1)
ax.set_title("Correlation matrix, reordered by the graph-cut partition\n(top-left block: diversified cluster)")
fig.colorbar(im, ax=ax, shrink=0.8, label="correlation")
plt.tight_layout()
plt.show()

## 6. Takeaways

- The **market graph** turns pairwise return correlation directly into Max-Cut edge weights;
  maximizing the cut pushes strongly-correlated pairs apart and leaves weakly- or
  negatively-correlated pairs together, so one side of the resulting partition is a more
  diversified basket than a same-sized random pick.
- **QAOA** solves this exactly the way it solves any graph Max-Cut problem, one qubit per
  node — fine for the 8-asset toy case, where it landed within a percent of the brute-force
  optimum, but it does not scale to a real investable universe.
- **PCE** solves the *same* objective on the *same* market graph, but encodes all 50 assets
  into just a handful of qubits via higher-order Pauli correlators, and got within a few
  percent of a classical local-search heuristic in seconds.
- The diversification benefit is real and consistent, but for a single two-way split it is
  modest by construction: Max-Cut on a same-sector clique only 2-colors it, so it separates
  roughly half of each sector's internal redundancy, not all of it. See
  `paper_thread1_dpce_vs_pce.ipynb` and the `paper_*_streaming_*` notebooks in this directory
  for the same idea pushed further: many clusters via recursive Max-Cut bipartition, streamed
  over a real rolling backtest.

## References

- Vicente P. Soloviev, Michal Krompiec, ["Large-scale portfolio optimization using Pauli
  Correlation Encoding"](https://arxiv.org/abs/2511.21305), arXiv:2511.21305 — the source
  paper for the PCE algorithm used in the large-scale example above.
- Vicente P. Soloviev, Antonio Márquez Romero, Josh Kirsopp, Michal Krompiec, ["Scaling
  Portfolio Diversification with Quantum Circuit Cutting
  Techniques"](https://arxiv.org/abs/2506.08947), arXiv:2506.08947 — quantum circuit
  cutting applied to the same portfolio-diversification problem this notebook solves with
  Max-Cut.
- Jacobo Padín-Martínez, Vicente P. Soloviev, Alejandro Borrallo-Rentero, Antón
  Rodríguez-Otero, Raquel Alfonso-Rodríguez, Michal Krompiec, ["Progressive
  Binarization -- Pauli Correlation Encoding: a Continuation Method for Constrained
  Optimization"](https://arxiv.org/abs/2602.17479), arXiv:2602.17479 — the
  iterative-alpha continuation scheme behind `qarp.algorithms.iterativePCE`, a
  natural next step from the single-shot `PCE` used above.